# Leaf Disease Detection - Standalone Training Notebook

Trains an EfficientNet-B0 classifier on **any** folder-based dataset (one subfolder per class). Uses the exact high-accuracy recipe from the HorticBot project: 2-phase training, mixed precision on GPU, augmentation, best-checkpoint saving, and real held-out test accuracy.

### Dataset layout
`
train/
  class_a/*.jpg
  class_b/*.jpg
test/             (optional)
  class_a/*.jpg
val/              (optional, else split from train)
`
Edit the TRAIN_DIR / TEST_DIR / VAL_DIR / MODEL_SAVE_PATH paths below.


In [ ]:

# ============================================================
#  CONFIG AND SETUP - edit these paths for YOUR dataset
# ============================================================
import os
import time
import random
from pathlib import Path
import torch  # noqa  (needed below for GPU auto-detection)

# ---- POINT THESE TO YOUR DATASET ----
# Each must contain one subfolder per class, e.g.
#   train/aphids/*.jpg, train/armyworm/*.jpg, ...
#   test/aphids/*.jpg  (optional but recommended for true accuracy)
TRAIN_DIR = "data/disease/PlantVillage"
TEST_DIR  = "data/disease/test"     # set to None if you have no test set
VAL_DIR   = "data/disease/val"      # set to None if you have no val set

MODEL_SAVE_PATH = "models/best_disease_model.pt"   # where to save the trained weights
MODEL_NAME = "efficientnet_b0"

# ---- Training hyper-parameters (tuned for best accuracy) ----
IMG_SIZE    = 224
BATCH_SIZE  = 32 if torch.cuda.is_available() else 16
HEAD_EPOCHS = 5
FT_EPOCHS   = 16
VAL_FRAC    = 0.10   # fraction of TRAIN held out as validation if VAL_DIR is None
SEED        = 42


In [ ]:

# ============================================================
#  IMPORTS
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(12)
torch.manual_seed(SEED)
random.seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
print("Using device:", device, "| torch", torch.__version__)


In [ ]:

# ============================================================
#  DATASET (auto-discovers class folders)
# ============================================================
class FolderDataset(Dataset):
    def __init__(self, root, transform=None, extensions=(".jpg",".jpeg",".png",".bmp")):
        self.root = Path(root)
        self.transform = transform
        self.classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.classes = [c for c in self.classes if not c.startswith((".", "_"))]
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.samples = []
        for cname in self.classes:
            cdir = self.root / cname
            files = sorted([f for f in cdir.iterdir()
                            if f.suffix.lower() in extensions and not f.name.startswith(".")])
            for f in files:
                self.samples.append((str(f), self.class_to_idx[cname]))
        if len(self.samples) == 0:
            raise RuntimeError(f"No images found under {root}")
        print(f"{self.root.name}: {len(self.samples)} images, {len(self.classes)} classes")
        for c in self.classes:
            cnt = sum(1 for _, y in self.samples if self.class_to_idx[c] == y)
            print(f"    {c}: {cnt}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        with open(path, "rb") as f:
            img = Image.open(f).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, y


def val_transform(size):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def train_transform(size):
    return transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(25),
        transforms.ColorJitter(brightness=0.25, contrast=0.25,
                               saturation=0.25, hue=0.08),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.15, scale=(0.02, 0.12)),
    ])


In [ ]:

# ============================================================
#  BUILD TRAIN / VAL / TEST LOADERS
# ============================================================
def make_loader(path, transform, batch_size, shuffle, pin):
    ds = FolderDataset(path, transform=transform)
    return ds, DataLoader(ds, batch_size=min(batch_size, max(1, len(ds))), shuffle=shuffle,
                          num_workers=0,
                          pin_memory=pin)


tr = train_transform(IMG_SIZE)
va = val_transform(IMG_SIZE)

pin = False  # Keep False on Windows for smooth execution without driver page lock delay

train_ds, train_loader = make_loader(TRAIN_DIR, tr, BATCH_SIZE, True, pin)

if VAL_DIR and os.path.isdir(VAL_DIR):
    val_ds, val_loader = make_loader(VAL_DIR, va, BATCH_SIZE, False, pin)
else:
    # hold out a val fraction from train
    n_val = int(len(train_ds) * VAL_FRAC)
    idx = list(range(len(train_ds)))
    random.seed(SEED); random.shuffle(idx)
    val_idx, tr_idx = set(idx[:n_val]), idx[n_val:]
    from torch.utils.data import Subset
    val_ds = Subset(train_ds, sorted(val_idx))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=pin)
    # re-create train loader without the val images
    train_ds2 = Subset(train_ds, tr_idx)
    train_loader = DataLoader(train_ds2, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=pin)

num_classes = len(train_ds.classes)
print("num_classes =", num_classes)
print("train batches =", len(train_loader), "| val batches =", len(val_loader))

test_loader = None
if TEST_DIR and os.path.isdir(TEST_DIR):
    _, test_loader = make_loader(TEST_DIR, va, BATCH_SIZE, False, pin)


In [ ]:

# ============================================================
#  MODEL - EfficientNet-B0 (change MODEL_NAME to switch backbone)
# ============================================================
def build_model(num_classes):
    if MODEL_NAME == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif MODEL_NAME == "efficientnet_b1":
        model = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown MODEL_NAME {MODEL_NAME}")
    return model.to(device)

model = build_model(num_classes)

# class-balanced loss (helps imbalanced datasets)
if hasattr(train_loader.dataset, "samples"):
    all_targets = [y for _, y in train_loader.dataset.samples]
elif hasattr(train_loader.dataset, "dataset") and hasattr(train_loader.dataset.dataset, "samples"):
    # Subset case
    all_targets = [train_loader.dataset.dataset.samples[i][1] for i in train_loader.dataset.indices]
else:
    all_targets = []

if all_targets:
    counts = torch.tensor([sum(1 for y in all_targets if y == i) for i in range(num_classes)], dtype=torch.float)
    class_weights = counts.sum() / (num_classes * counts.clamp(min=1))
    class_weights = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)


In [ ]:

# ============================================================
#  TRAINING LOOP (with AMP + progress)
# ============================================================
def run_epoch(model, loader, criterion, optimizer, scaler, split):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    n_total = len(loader.dataset)
    t0 = time.time()
    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        if False:
            pass
        else:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
    return running_loss / max(total, 1), 100.0 * correct / max(total, 1), total


def evaluate(model, loader, split="val"):
    model.eval()
    correct = 0
    total = 0
    n_total = len(loader.dataset)
    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    return 100.0 * correct / max(total, 1)


scaler = None  # AMP scaler disabled for smooth cross-platform execution
best_acc = 0.0
best_state = None

# ---- Phase 1: train head only ----
for p in model.features.parameters():
    p.requires_grad = False
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
print(f"\n[Phase 1/2] Training classifier head ({HEAD_EPOCHS} epochs)...")
for epoch in range(HEAD_EPOCHS):
    loss, tacc, n = run_epoch(model, train_loader, criterion, optimizer, scaler, "train")
    vacc = evaluate(model, val_loader)
    print(f"  head epoch {epoch+1}/{HEAD_EPOCHS} | loss {loss:.4f} | train {tacc:.2f}% | val {vacc:.2f}%", flush=True)
    if vacc >= best_acc:
        best_acc = vacc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# ---- Phase 2: fine-tune full network ----
for p in model.features.parameters():
    p.requires_grad = True
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=5e-5)
sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS, eta_min=1e-6)
print(f"\n[Phase 2/2] Fine-tuning full network ({FT_EPOCHS} epochs)...", flush=True)
for epoch in range(FT_EPOCHS):
    loss, tacc, n = run_epoch(model, train_loader, criterion, optimizer, scaler, "train")
    vacc = evaluate(model, val_loader)
    sched.step()
    print(f"  ft epoch {epoch+1}/{FT_EPOCHS} | loss {loss:.4f} | train {tacc:.2f}% | val {vacc:.2f}% | lr {optimizer.param_groups[0]['lr']:.2e}", flush=True)
    if vacc >= best_acc:
        best_acc = vacc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest validation accuracy: {best_acc:.2f}%")
if best_state is not None:
    model.load_state_dict(best_state)
    os.makedirs(os.path.dirname(MODEL_SAVE_PATH) or ".", exist_ok=True)
    torch.save(best_state, MODEL_SAVE_PATH)
    print(f"Saved best model -> {MODEL_SAVE_PATH}")


In [ ]:

# ============================================================
#  HELD-OUT TEST ACCURACY + CONFIGURATION MATRIX
# ============================================================
if test_loader is not None:
    test_acc = evaluate(model, test_loader, "test")
    print(f"\n>>> HELD-OUT TEST accuracy: {test_acc:.2f}% on {len(test_loader.dataset)} images")
    # per-class breakdown
    model.eval()
    per = {}
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device); targets = targets.to(device)
            _, preds = torch.max(model(inputs), 1)
            for t, p in zip(targets.tolist(), preds.tolist()):
                per.setdefault(t, [0, 0])
                per[t][1] += 1
                if t == p:
                    per[t][0] += 1
    cls = test_loader.dataset.classes
    for i in sorted(per):
        correct_i, total_i = per[i]
        print(f"    {cls[i]:40s} {correct_i}/{total_i}  ({100.0*correct_i/max(total_i,1):.1f}%)")
else:
    print("\nNo test set provided. Skipping held-out test accuracy.")


In [ ]:

# ============================================================
#  LOAD SAVED MODEL + PREDICT A SINGLE IMAGE
# ============================================================
def load_predictor(path, num_classes):
    m = build_model(num_classes)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    m.eval()
    return m

def predict_image(model, image_path, class_list):
    img = Image.open(image_path).convert("RGB")
    x = val_transform(IMG_SIZE)(img).unsqueeze(0)
    with torch.no_grad():
        logits = model(x.to(device))
        prob = torch.softmax(logits, dim=1)[0]
    idx = int(prob.argmax())
    return class_list[idx], float(prob[idx]) * 100.0, prob.cpu().numpy()

# Example: predict all test images and count correct
if test_loader is not None:
    model2 = load_predictor(MODEL_SAVE_PATH, num_classes)
    class_list = test_loader.dataset.classes
    import glob
    print("\nSample predictions (first few test images):")
    shown = 0
    for c in class_list:
        for f in sorted(glob.glob(os.path.join(TEST_DIR, c, "*")))[:2]:
            if not f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                continue
            pred, conf, _ = predict_image(model2, f, class_list)
            print(f"  true={c:30s} pred={pred:30s} conf={conf:.1f}%  {'OK' if c==pred else 'MISS'}")
            shown += 1


## Done

Your best model is saved at `models/best_disease_model.pt`.

- **val accuracy** and (if a test set was given) **held-out test accuracy** are printed above.
- Change `TRAIN_DIR`, `TEST_DIR`, `VAL_DIR`, and `MODEL_SAVE_PATH` to train on any dataset.
- Raise `FT_EPOCHS` for more accuracy (costs time).
- On GPU, mixed precision is used automatically; on CPU it falls back to fp32.
